# MemGraphRAG Uygulamasının Google Colab'da Çalıştırılması

**Ders:** YZM313 – Yapay Zekâ (Warm-up Ödevi)
**Öğrenci:** _Ad Soyad / Öğrenci No_
**Kullanılan dil modeli:** **Qwen3.5-9B** (`qwen3.5:9b`, açık kaynak, Colab T4 GPU'sunda Ollama ile yerel)
**Embedding modeli:** `BAAI/bge-large-en-v1.5` (Colab GPU'sunda yerel)
**Örnek veri seti:** Repodaki `dataset/hotpotqa` (çok adımlı soru-cevap) – 10 soruluk alt küme

### İçindekiler
1. MemGraphRAG'ın temel çalışma mantığı
2. Dil modeli seçimi ve gerekçesi
3. Ortam hazırlığı (GPU, repo, kütüphaneler)
4. Dil modelinin Colab'da kurulması (Ollama) ve test edilmesi
5. Modelin MemGraphRAG'a entegrasyonu (kod değişiklikleri)
6. Veri setinin hazırlanması
7. İndeksleme (bellek ve graf oluşturma)
8. Oluşan üç katmanlı belleğin incelenmesi
9. Erişim (retrieval) ve soru-cevap
10. Klasik RAG ile karşılaştırma
11. Karşılaşılan hatalar ve çözümleri
12. Teknik değerlendirme: MemGraphRAG vs. klasik RAG vs. GraphRAG

> **Çalıştırma:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU` seçildikten sonra hücreler sırayla çalıştırılmalıdır.
> API anahtarı veya ücretli hesap gerekmez; model Colab'ın GPU'sunda çalışır.

## 1. MemGraphRAG'ın Temel Çalışma Mantığı

MemGraphRAG (KDD'26, [arXiv:2606.00610](https://arxiv.org/abs/2606.00610)), HippoRAG üzerine kurulu, **bellek (memory) tabanlı** bir GraphRAG yöntemidir.
Klasik RAG metin parçalarını doğrudan vektör benzerliğiyle getirirken, MemGraphRAG bilgiyi önce **üç katmanlı bir belleğe** dönüştürür:

| Katman | İçerik | Örnek |
|---|---|---|
| **Şema (Schema) katmanı** | Soyut ontoloji üçlüleri `(baş tipi, ilişki, kuyruk tipi)` | `(<Person>, directed, <Film>)` |
| **Olgu (Fact) katmanı** | Metinden çıkarılmış somut üçlüler `(özne, ilişki, nesne)` | `(Andrew Stanton, directed, Finding Nemo)` |
| **Pasaj (Passage) katmanı** | Olguların dayandığı orijinal metin parçaları | `"Finding Nemo is a 2003 film directed by..."` |

Katmanlar çift yönlü bağlantılıdır: her olgu hem kendi şemasına hem de kaynak pasajına bağlıdır.

### İndeksleme hattı (pipeline) – `MemGraphRAG.index_with_memory()`
1. **Parçalama (chunking):** Korpus, embedding modelinin tokenizer'ıyla 256 token'lık (32 token örtüşmeli) parçalara bölünür.
2. **OpenIE (açık bilgi çıkarımı):** LLM her parça için önce varlıkları (NER), sonra `(özne, ilişki, nesne)` üçlülerini çıkarır.
3. **Ontoloji çıkarımı:** LLM her olguyu, pasaj bağlamına bakarak bir şema üçlüsüne soyutlar → şema katmanı oluşur.
4. **Ontoloji filtreleme:** En düşük frekanslı %20 şema (ve bunlara bağlı gürültülü olgular) elenir.
5. **Çelişki tespiti:** Ortak varlığı olan olgular aday çiftler olur; LLM pasaj kanıtıyla "katı çelişki" olup olmadığına karar verir (güven ≥ 0.85).
6. **Çelişki çözümü:** Birbirine bağlı çelişki grupları LLM'e verilir; her olgu için *koru / düzelt / at* kararı uygulanır.
7. **Bellekten graf üretimi:** Temizlenmiş bellekten **tip, varlık ve pasaj düğümleri** içeren bir graf kurulur; anlamca benzer varlıklar arasına eş anlamlılık (synonymy) kenarları eklenir (benzerlik ≥ 0.8).

### Sorgulama hattı – `MemGraphRAG.retrieve()` + `qa()`
1. Soru embedding'e çevrilir; tüm olgularla benzerlik skorları hesaplanır ve eşik (0.4) altındakiler atılır.
2. Kalan olguların varlıkları **tohum (seed) düğüm** olur; pasaj düğümlerine de yoğun (dense) benzerlik skoruyla küçük bir ağırlık verilir.
3. Graf üzerinde **Kişiselleştirilmiş PageRank (Personalized PageRank – PPR)** çalıştırılır; en yüksek skorlu pasajlar getirilir.
4. Getirilen pasajlar ve soru LLM'e verilir, LLM "Answer:" ile biten bir cevap üretir.

Kısaca: MemGraphRAG'ın farkı, grafı ham OpenIE çıktısından değil **şemalarla düzenlenmiş ve çelişkilerden arındırılmış bir bellekten** üretmesidir.

## 2. Dil Modeli Seçimi: Qwen3.5-9B

**Kullanılan model:** `Qwen3.5-9B` – Alibaba Qwen ekibinin Mart 2026'da yayımladığı Qwen3.5 serisinin 9 milyar parametreli modeli.
Ağırlıkları **Apache 2.0** lisansıyla açıktır. Colab'da **Ollama** aracılığıyla 4-bit sıkıştırılmış (Q4, ~6.6 GB) haliyle çalıştırılıyor.

### Ödevde verilen kriterlere göre değerlendirme

| Kriter | Değerlendirme |
|---|---|
| **Colab'da çalıştırılabilirlik** | Ücretsiz T4 GPU'nun 15 GB belleğine sığıyor (aşağıdaki bellek hesabı). Ollama tek komutla kuruluyor. |
| **MemGraphRAG ile uyum** | Repo, LLM'e OpenAI uyumlu API ile bağlanıyor (`code/src/llm/openai_gpt.py`). Ollama da aynı formatta bir sunucu açıyor (`http://localhost:11434/v1`). Bu yüzden kodda yalnızca model adı ve adres değişiyor. Repo, `localhost` adresinde API anahtarı istemiyor. |
| **GPU / RAM gereksinimi** | Model ~6.6 GB + bağlam belleği (4 paralel istek × 8192 token) ~3–4 GB + embedding modeli ~1.3 GB ≈ **11–12 GB**. 15 GB'lık T4'te pay kalıyor. Model işlemciye taşmadığı için hız korunuyor. |
| **Cevap kalitesi** | MemGraphRAG'ın her aşaması LLM'den **geçerli JSON** bekliyor, bu yüzden talimat takibi en önemli ölçüt. Qwen3.5-9B, IFEval (talimat takibi) testinde %91.5, MMLU-Pro (genel bilgi/akıl yürütme) testinde %82.5 alıyor. Aynı boyuttaki Gemma 4 E4B MMLU-Pro'da %69.4'te kalıyor. |
| **Maliyet** | Ücretsiz. API ücreti, anahtar veya kredi kartı gerekmiyor. Bedeli süre: ~500 LLM çağrısı T4'te yaklaşık 20–30 dakika sürüyor. |
| **Açık kaynak / API** | Açık ağırlıklı (Apache 2.0); model indirilip tamamen yerelde çalışıyor, veri dışarı gönderilmiyor. |
| **Güncellik** | Değerlendirilen ve T4'e sığan modeller arasında en yeni nesil (2026) ve en yüksek skorlu model. |

**Düşünme modu:** Qwen3.5 ailesinin büyük modelleri cevaptan önce uzun "düşünce" metni üretiyor. Alibaba, küçük modellerde (0.8B–9B) bu modu **varsayılan olarak kapalı** getiriyor. Bu bizim için önemli: düşünme metni hem 500 çağrıda süreyi katlıyor hem de JSON çıktısını bozabiliyor.

### Değerlendirilen alternatifler

| Model | Neden seçilmedi |
|---|---|
| **Gemma 4 E4B** (Nisan 2026) | Colab T4'te çalıştığı raporlanmış, ama kalite skorları daha düşük ve düşünme modu varsayılan açık. **Yedek model olarak** ayarlandı. |
| Ministral 3 8B, Llama 3.1 8B, Granite 4.1 8B | T4'e rahat sığıyorlar, ama kalite skorları Qwen3.5-9B'nin gerisinde. |
| Qwen3-14B, Ministral 3 14B, Phi-4 14B | T4'te yalnızca 2 paralel istek sığıyor, bu da indekslemeyi yaklaşık 2 kat yavaşlatıyor. |
| gpt-oss-20b, Gemma 4 26B/31B, Qwen3.6 27B | T4'ün 15 GB belleğine sığmıyor; işlemciye taşınca saniyede 5–10 token'a düşüyor. |
| API modelleri (GPT-4o-mini, MiMo vb.) | Ücretli hesap ve API anahtarı gerektiriyor. |

### Bilinen risk ve önlem
Ollama'da Qwen3.5'in T4 ile aynı mimarideki (Turing) bazı kartlarda çöktüğüne dair bir hata kaydı var ([ollama#14715](https://github.com/ollama/ollama/issues/14715)); Colab'da sorunsuz çalıştığı raporları da var.
Bu yüzden Bölüm 4.4'te model, indekslemeden önce GPU üzerinde JSON üreterek test ediliyor. Hata verirse ya da GPU yerine işlemcide çalışırsa, tek satır değiştirilerek **Gemma 4 E4B**'ye geçiliyor.

## 3. Ortam Hazırlığı
### 3.1 GPU kontrolü

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, sys
print("Python:", sys.version.split()[0], "| torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

### 3.2 Reponun klonlanması

In [ ]:
%cd /content
!git clone -q https://github.com/XMUDeepLIT/MemGraphRAG.git
%cd /content/MemGraphRAG
!git log --oneline -1
!ls

### 3.3 Kütüphanelerin kurulumu

Repodaki `requirements.txt` doğrudan kurulmadı, çünkü:
- `torch==2.5.1` ve `transformers==4.45.2` sabitlenmiş. Colab'ın hazır torch/CUDA sürümünü düşürüp uyumsuzluk çıkarıyor ve kurulumu çok uzatıyor.
- `vllm==0.6.6.post1` yalnızca **çevrimdışı (offline) OpenIE** için gerekiyor. Biz online modu (OpenAI uyumlu sunucu olarak Ollama) kullanıyoruz. Ayrıca vllm Colab'da en çok kurulum sorunu çıkaran paket ve T4'te bazı özellikleri desteklemiyor.

Bu yüzden Colab'daki torch korunup kodun gerçekten import ettiği paketler kuruldu.
`gritlm` paketi kodda koşulsuz import edildiği için kuruldu (kullanılmasa da, yoksa `ModuleNotFoundError` veriyor).

In [ ]:
!pip install -q "transformers>=4.45,<5" python-igraph openai tenacity tiktoken filelock einops nest_asyncio gritlm
import transformers, igraph, openai
print("transformers", transformers.__version__, "| igraph", igraph.__version__, "| openai", openai.__version__)

## 4. Dil Modelinin Colab'da Kurulması (Ollama) ve Test Edilmesi

**Ollama**, açık kaynak modelleri indirip GPU'da çalıştıran ve onları OpenAI formatında bir sunucu olarak açan bir araç. MemGraphRAG bu sunucuya, OpenAI'ın API'sine bağlanır gibi bağlanıyor.

### 4.1 Ollama kurulumu
Colab'da iki noktaya dikkat edildi:
- Ollama'nın güncel kurulum paketi `zstd` ile sıkıştırılmış; Colab'da bu araç yok, önce o kuruluyor.
- Colab'da `systemd` olmadığı için Ollama servis olarak başlamıyor; sunucu bir sonraki hücrede elle, arka planda başlatılıyor (kurulum sırasındaki systemd uyarıları normaldir).

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

### 4.2 Ollama sunucusunun başlatılması
Sunucu, T4'e ve MemGraphRAG'ın ihtiyacına göre ayarlanarak başlatılıyor:
- `OLLAMA_CONTEXT_LENGTH=8192`: Ollama varsayılan olarak modele en fazla 4096 token gösteriyor ve fazlasını **hata vermeden sessizce kesiyor**. MemGraphRAG'ın çelişki tespiti istemleri bu sınırı aşabildiği için 8192'ye çıkarıldı.
- `OLLAMA_NUM_PARALLEL=4`: Aynı anda 4 istek işleniyor. Bellek hesabına göre T4'e sığan en verimli değer.
- `OLLAMA_KEEP_ALIVE=-1`: Model iki istek arasında bellekten atılmıyor.

In [ ]:
import os, subprocess, time, requests

ollama_env = dict(os.environ,
                  OLLAMA_CONTEXT_LENGTH="8192",
                  OLLAMA_NUM_PARALLEL="4",
                  OLLAMA_KEEP_ALIVE="-1",
                  OLLAMA_MAX_LOADED_MODELS="1")
ollama_log = open("/content/ollama.log", "w")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=ollama_env, stdout=ollama_log, stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        print("Ollama sunucusu hazır, sürüm:", requests.get("http://localhost:11434/api/version", timeout=2).json()["version"])
        break
    except requests.exceptions.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Ollama başlamadı, /content/ollama.log dosyasına bakın")

### 4.3 Model ayarları ve modelin indirilmesi
Yedek modele geçmek gerekirse yalnızca `LLM_NAME` satırı değiştirilip 4.3 ve 4.4 hücreleri yeniden çalıştırılır.

In [ ]:
import json

# --- Model ayarları (indeksleme ve sorgulamada AYNI olmalı; klasör adları bunlardan türetiliyor) ---
LLM_NAME        = "qwen3.5:9b"        # yedek: "gemma4:e4b"
LLM_BASE_URL    = "http://localhost:11434/v1"
EMBEDDING_MODEL = "BAAI/bge-large-en-v1.5"

# Modele özel ek istek parametreleri: düşünme modu küçük Qwen3.5'lerde zaten kapalı,
# Gemma 4'te varsayılan açık olduğu için kapatılıyor.
MODEL_EXTRA_BODY = {"qwen3.5:9b": {}, "gemma4:e4b": {"reasoning_effort": "none"}}
LLM_EXTRA_BODY   = MODEL_EXTRA_BODY.get(LLM_NAME, {})

os.environ["OPENAI_API_KEY"] = "ollama"            # Ollama anahtar istemez; OpenAI istemcisi boş değer kabul etmediği için
os.environ["LLM_EXTRA_BODY"] = json.dumps(LLM_EXTRA_BODY)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!ollama pull {LLM_NAME} 2>&1 | tail -n 2
!ollama list

### 4.4 Modelin test edilmesi (JSON üretimi, hız ve GPU kontrolü)
MemGraphRAG'ın ilk adımına (varlık çıkarma) benzer bir istekle modelin **geçerli JSON** ürettiği, hızı ve **tamamen GPU'da** çalıştığı kontrol ediliyor.
`ollama ps` çıktısındaki `PROCESSOR` sütunu **`100% GPU`** olmalı. Model hata verirse ya da işlemcide (CPU) çalışıyorsa 4.3'te `LLM_NAME = "gemma4:e4b"` yapılmalı.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=LLM_BASE_URL)
messages = [
    {"role": "system", "content": 'Extract the named entities from the paragraph. Respond only with JSON: {"named_entities": [...]}'},
    {"role": "user", "content": "The Newcomers is a 2000 family drama film directed by James Allen Bradley and starring Christopher McCoy, Kate Bosworth and Chris Evans."},
]
try:
    client.chat.completions.create(model=LLM_NAME, messages=messages, max_tokens=5)  # ısınma: modeli GPU'ya yükler
    start = time.time()
    resp = client.chat.completions.create(model=LLM_NAME, messages=messages, max_tokens=300,
                                          temperature=0, extra_body=LLM_EXTRA_BODY or None)
    elapsed = time.time() - start
    content = resp.choices[0].message.content or ""
    print("Cevap :", content)
    try:
        ok = "named_entities" in json.loads(content[content.find("{"):content.rfind("}") + 1])
    except ValueError:
        ok = False
    print("JSON  :", "geçerli ✓" if ok else "HATALI ✗ (model JSON üretemedi; yedek modeli deneyin)")
    print(f"Hız   : {resp.usage.completion_tokens / elapsed:.1f} token/sn ({resp.usage.completion_tokens} token, {elapsed:.1f} sn)")
except Exception as exc:
    print("MODEL HATA VERDİ:", exc)
    print("→ 4.3'te LLM_NAME = 'gemma4:e4b' yapıp 4.3 ve 4.4'ü tekrar çalıştırın. Ayrıntı: /content/ollama.log")
!ollama ps
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 5. Modelin MemGraphRAG'a Entegrasyonu

MemGraphRAG tüm LLM çağrılarını `CacheOpenAI` sınıfı (`code/src/llm/openai_gpt.py`) üzerinden yapıyor. Yerel modelle (Ollama) güvenilir çalışması için bu dosyada üç değişiklik yapıldı (değişen satırlar `Model entegrasyonu` yorumuyla işaretli):

1. **Modele özel parametre (`extra_body`):** `LLM_EXTRA_BODY` ortam değişkenindeki ayar (ör. yedek modelde düşünme modunu kapatma) her isteğe ekleniyor.
2. **Tekrar deneme (retry):** Orijinal kodda tek bir geçici hata (sunucu meşgul, zaman aşımı, 5xx) bütün indekslemeyi durduruyordu. `tenacity` ile üstel bekleme (exponential backoff) kullanan 6 denemelik bir tekrar mekanizması eklendi.
3. **Cevap temizliği:** Boş (`None`) cevaplar `""` yapılıyor; `<think>...</think>` blokları JSON ayrıştırmadan önce siliniyor.

Geri kalan her şey (önbellek, token sayımı, `max_tokens` dönüşümü) orijinal haliyle duruyor. Model adı `gpt` içermediği için kod `max_completion_tokens` yerine `max_tokens` gönderiyor; Ollama bunu destekliyor.

In [ ]:
%%writefile code/src/llm/openai_gpt.py
import functools
import re
import hashlib
import json
import os
import sqlite3
from copy import deepcopy
from typing import List, Tuple

import httpx
import openai
from filelock import FileLock
from openai import OpenAI
from packaging import version
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential

from ..utils.config_utils import BaseConfig
from ..utils.llm_utils import (
    TextChatMessage
)
from ..utils.logging_utils import get_logger
from .base import BaseLLM, LLMConfig

logger = get_logger(__name__)

def cache_response(func):
    @functools.wraps(func)
    def wrapper(self, *args, **kwargs):
        # get messages from args or kwargs
        if args:
            messages = args[0]
        else:
            messages = kwargs.get("messages")
        if messages is None:
            raise ValueError("Missing required 'messages' parameter for caching.")

        # get model, seed and temperature from kwargs or self.llm_config.generate_params
        gen_params = getattr(self, "llm_config", {}).generate_params if hasattr(self, "llm_config") else {}
        model = kwargs.get("model", gen_params.get("model"))
        seed = kwargs.get("seed", gen_params.get("seed"))
        temperature = kwargs.get("temperature", gen_params.get("temperature"))

        # build key data, convert to JSON string and hash to generate key_hash
        key_data = {
            "messages": messages,  # messages requires JSON serializable
            "model": model,
            "seed": seed,
            "temperature": temperature,
        }
        key_str = json.dumps(key_data, sort_keys=True, default=str)
        key_hash = hashlib.sha256(key_str.encode("utf-8")).hexdigest()

        # the file name of lock, ensure mutual exclusion when accessing concurrently
        lock_file = self.cache_file_name + ".lock"

        # Try to read from SQLite cache
        with FileLock(lock_file):
            conn = sqlite3.connect(self.cache_file_name)
            c = conn.cursor()
            # if the table does not exist, create it
            c.execute("""
                CREATE TABLE IF NOT EXISTS cache (
                    key TEXT PRIMARY KEY,
                    message TEXT,
                    metadata TEXT
                )
            """)
            conn.commit()  # commit to save the table creation
            c.execute("SELECT message, metadata FROM cache WHERE key = ?", (key_hash,))
            row = c.fetchone()
            conn.close()
            if row is not None:
                message, metadata_str = row
                metadata = json.loads(metadata_str)
                # return cached result and mark as hit
                return message, metadata, True

        # if cache miss, call the original function to get the result
        result = func(self, *args, **kwargs)
        message, metadata = result

        # insert new result into cache
        with FileLock(lock_file):
            conn = sqlite3.connect(self.cache_file_name)
            c = conn.cursor()
            # make sure the table exists again (if it doesn't exist, it would be created)
            c.execute("""
                CREATE TABLE IF NOT EXISTS cache (
                    key TEXT PRIMARY KEY,
                    message TEXT,
                    metadata TEXT
                )
            """)
            metadata_str = json.dumps(metadata)
            c.execute("INSERT OR REPLACE INTO cache (key, message, metadata) VALUES (?, ?, ?)",
                      (key_hash, message, metadata_str))
            conn.commit()
            conn.close()

        return message, metadata, False

    return wrapper


# Model entegrasyonu: gecici hatalarda (sunucu mesgul, zaman asimi, 5xx) tekrar dene.
# Orijinal kodda tek bir hata tum indekslemeyi durduruyordu.
RETRYABLE_ERRORS = (openai.RateLimitError, openai.APIConnectionError,
                    openai.APITimeoutError, openai.InternalServerError)

# Dusunme (reasoning) modu acik kalirsa bazi modeller cevabin basina <think>...</think> ekler;
# JSON ayristirmayi bozmamasi icin temizlenir.
THINK_PATTERN = re.compile(r"<think>.*?</think>", re.DOTALL)


class CacheOpenAI(BaseLLM):
    """OpenAI LLM implementation."""
    @classmethod
    def from_experiment_config(cls, global_config: BaseConfig) -> "CacheOpenAI":
        config_dict = global_config.__dict__
        cache_dir = os.path.join(global_config.save_dir, "llm_cache")
        return cls(cache_dir=cache_dir, **config_dict)

    def __init__(self, cache_dir, cache_filename: str = None,
                 llm_name: str = "gpt-4o-mini", api_key: str = None, llm_base_url: str = None, 
                 high_throughput: bool = True,
                 **kwargs) -> None:
        super().__init__()
        self.cache_dir = cache_dir
        os.makedirs(self.cache_dir, exist_ok=True)
        if cache_filename is None:
            cache_filename = f"{llm_name.replace('/', '_')}_cache.sqlite"
        self.cache_file_name = os.path.join(self.cache_dir, cache_filename)
        self.llm_name = llm_name
        self.llm_base_url = llm_base_url
        # Model entegrasyonu: modele ozel ek parametreler (or. {"reasoning_effort": "none"})
        self.extra_body = json.loads(os.getenv("LLM_EXTRA_BODY", "{}") or "{}")
        self._init_llm_config(**kwargs)
        if high_throughput:
            limits = httpx.Limits(max_connections=500, max_keepalive_connections=100)
            client = httpx.Client(limits=limits, timeout=httpx.Timeout(5*60, read=5*60))
        else:
            client = None
        self.openai_client = OpenAI(base_url=self.llm_base_url, api_key=api_key, http_client=client)

    def _init_llm_config(self, **kwargs) -> None:
        config_dict = {
            "llm_name": self.llm_name,
            "llm_base_url": self.llm_base_url,
            "generate_params": {
                "model": self.llm_name,
                "max_completion_tokens": kwargs.get("max_new_tokens", 400),
                "n": kwargs.get("num_gen_choices", 1),
                "seed": kwargs.get("seed", 0),
                "temperature": kwargs.get("temperature", 0.0),
            }
        }
        self.llm_config = LLMConfig.from_dict(config_dict=config_dict)
        logger.debug(f"Init {self.__class__.__name__}'s llm_config: {self.llm_config}")

    @cache_response
    def infer(
        self,
        messages: List[TextChatMessage],
        **kwargs
    ) -> Tuple[List[TextChatMessage], dict]:
        params = deepcopy(self.llm_config.generate_params)
        if kwargs:
            params.update(kwargs)
        params["messages"] = messages
        logger.debug(f"Calling OpenAI GPT API with:\n{params}")

        if 'gpt' not in params['model'] or version.parse(openai.__version__) < version.parse("1.45.0"): # if we use vllm to call openai api or if we use openai but the version is too old to use 'max_completion_tokens' argument
            # TODO strange version change in openai protocol, but our current vllm version not changed yet
            params['max_tokens'] = params.pop('max_completion_tokens')

        if self.extra_body:
            params["extra_body"] = self.extra_body

        response = self._create_with_retry(params)

        response_message = response.choices[0].message.content or ""
        response_message = THINK_PATTERN.sub("", response_message).strip()

        metadata = {
            "prompt_tokens": response.usage.prompt_tokens, 
            "completion_tokens": response.usage.completion_tokens,
            "finish_reason": response.choices[0].finish_reason,
        }

        return response_message, metadata

    @retry(retry=retry_if_exception_type(RETRYABLE_ERRORS), stop=stop_after_attempt(6),
           wait=wait_exponential(multiplier=2, min=2, max=60), reraise=True)
    def _create_with_retry(self, params):
        return self.openai_client.chat.completions.create(**params)

Ek olarak bellek aşamalarındaki JSON ayrıştırıcı (`_memory_json`) bozuk bir LLM çıktısında istisna (exception) fırlatıp **tüm indekslemeyi** iptal ediyordu (`index_with_memory failed during schema extraction`).
Bu fonksiyon, onaramadığı tek bir cevabı uyarı verip boş sonuç (`{}`) olarak döndürecek şekilde değiştirildi. Böylece o tek olgu atlanıyor, hat devam ediyor.

In [ ]:
from pathlib import Path

path = Path("code/src/MemGraphRAG.py")
src = path.read_text()
old = "            return json.loads(fix_broken_generated_json(candidate))\n"
new = ("            try:\n"
       "                return json.loads(fix_broken_generated_json(candidate))\n"
       "            except Exception:\n"
       "                logger.warning(\"Memory JSON could not be parsed, skipping: %s\", cleaned[:200])\n"
       "                return {}\n")
if new in src:
    print("Yama zaten uygulanmış.")
else:
    assert src.count(old) == 1, "Beklenen satır bulunamadı"
    path.write_text(src.replace(old, new))
    print("_memory_json yaması uygulandı.")

## 6. Veri Setinin Hazırlanması

README'deki örnek komutlar `datasets/corpus/corpus.txt` ve `small-questions.json` dosyalarını kullanıyor, ancak bu dosyalar repoda yok (klasörün adı da `datasets/` değil `dataset/`).
Repoda gelen örnek veri setlerinden **HotpotQA** seçildi:

- **Neden HotpotQA?** Çok adımlı (multi-hop) sorular içeriyor: cevap için iki farklı paragraftaki bilgiyi birleştirmek gerekiyor. Graf tabanlı yöntemlerin klasik RAG'a göre avantajını göstermek için uygun bir test. Cevaplar kısa olduğu için Exact Match / F1 ile ölçülebiliyor.
- **Neden alt küme?** MemGraphRAG her olgu için ayrı LLM çağrısı yapıyor. 1000 sorunun tam korpusu (≈9800 paragraf) on binlerce çağrı demek. Colab T4'te indeksleme süresini ~20–30 dakikada tutmak için ilk **10 soru** alındı (`N_QUESTIONS` değiştirilerek artırılabilir; 20 soruda süre yaklaşık iki katına çıkar). Her soru için 2 destek paragrafı ve 2 çeldirici (ilgisiz) paragraf korpusa eklendi, böylece sistemin doğru paragrafı ayırt etmesi gerekiyor.

In [ ]:
import json
import os
import random

N_QUESTIONS = 10          # kac HotpotQA sorusu kullanilacak
N_DISTRACTORS = 2         # her soru icin eklenecek "celdirici" (ilgisiz) paragraf sayisi
DATA_DIR = "data/hotpotqa_small"

random.seed(42)
with open("dataset/hotpotqa/hotpotqa.json", encoding="utf-8") as f:
    hotpot = json.load(f)

selected = hotpot[:N_QUESTIONS]
paragraphs, seen_titles = [], set()
for sample in selected:
    support_titles = {title for title, _ in sample["supporting_facts"]}
    context = {title: sentences for title, sentences in sample["context"]}
    distractors = [t for t in context if t not in support_titles]
    chosen = sorted(support_titles) + random.sample(distractors, min(N_DISTRACTORS, len(distractors)))
    for title in chosen:
        if title in seen_titles or title not in context:
            continue
        seen_titles.add(title)
        paragraphs.append(f"{title}\n{''.join(context[title]).strip()}")

random.shuffle(paragraphs)  # destek paragraflari yan yana durmasin
questions = [
    {"_id": s["_id"], "question": s["question"].strip(), "answer": s["answer"],
     "type": s["type"], "supporting_facts": s["supporting_facts"]}
    for s in selected
]

os.makedirs(DATA_DIR, exist_ok=True)
with open(f"{DATA_DIR}/corpus.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(paragraphs))
with open(f"{DATA_DIR}/questions.json", "w", encoding="utf-8") as f:
    json.dump(questions, f, ensure_ascii=False, indent=2)

print(f"Paragraf sayisi : {len(paragraphs)}")
print(f"Korpus boyutu   : {sum(len(p) for p in paragraphs):,} karakter")
print(f"Soru sayisi     : {len(questions)}  (bridge: {sum(q['type'] == 'bridge' for q in questions)}, "
      f"comparison: {sum(q['type'] == 'comparison' for q in questions)})")
print("\nOrnek paragraf:\n" + paragraphs[0][:400])

## 7. İndeksleme: Bellek ve Graf Oluşturma

README'deki `code/index.py` komutu kendi veri setimiz ve modelimizle çalıştırılıyor.
- `--memory-max-workers 4`: Ollama'nın aynı anda işlediği istek sayısıyla (`OLLAMA_NUM_PARALLEL=4`) eşitlendi. Varsayılan 20 paralel istek T4'te yalnızca kuyrukta bekleyip zaman aşımına yol açardı.
- `--conflict-max-related 5`: Çelişki tespitinde her olgu varsayılan olarak 20 ilişkili olguyla ve onların kanıt pasajlarıyla birlikte modele veriliyor (~12.000 token). Bu, 8192 token'lık bağlama sığmayıp sessizce kesileceği için 5'e düşürüldü.
- `--artifact-mode detailed`: Ara çıktılar (çelişki tespiti/çözümü dosyaları) da kaydediliyor, Bölüm 8'de inceleniyor.

**Gözlem:** `index.py`, korpusu embedding modelinin tokenizer'ıyla token'lara bölüp tekrar metne çeviriyor (`tokenizer.decode`). `bge-large-en-v1.5` büyük/küçük harf ayırmayan (uncased) bir model olduğu için LLM'e giden parçalar **tamamen küçük harfli** oluyor ve noktalama işaretlerinin etrafına boşluk ekleniyor (ör. `chris evans ( actor )`). Sistem bu haliyle de çalışıyor; repodaki davranışı değiştirmemek için olduğu gibi bırakıldı. Ancak özel isimlerin büyük harf bilgisi kaybolduğu için çıkarılan varlıkların kalitesini bir miktar düşürebilir.

İlk çalıştırmada embedding modeli (~1.3 GB) HuggingFace'ten indiriliyor. LLM cevapları `outputs/.../llm_cache` altında önbelleğe alındığı için hücre yarıda kesilip tekrar çalıştırılırsa tamamlanan çağrılar tekrarlanmıyor. Qwen3.5-9B ile T4'te bu adım yaklaşık **20–30 dakika** sürüyor; Colab'ın boşta kalma süresi (~90 dk) dolmasın diye sekme açık tutulmalı.

In [ ]:
SAVE_DIR = "outputs/hotpotqa_small"
index_start = time.time()
!python code/index.py \
  --corpus data/hotpotqa_small/corpus.txt \
  --save-dir {SAVE_DIR} \
  --llm-name {LLM_NAME} \
  --llm-base-url {LLM_BASE_URL} \
  --embedding-model {EMBEDDING_MODEL} \
  --chunk-size 256 --chunk-overlap 32 \
  --memory-max-workers 4 \
  --conflict-max-related 5 \
  --artifact-mode detailed 2>&1 | grep -vE "httpx|it/s\]|s/it\]|^\s*$"
print(f"İndeksleme süresi: {(time.time() - index_start) / 60:.1f} dakika")

In [ ]:
!find {SAVE_DIR} -maxdepth 2 -not -path "*/llm_cache*" | sort

## 8. Oluşan Üç Katmanlı Belleğin İncelenmesi
### 8.1 Katman istatistikleri ve aşamaların etkisi

In [ ]:
import json, pandas as pd
from collections import Counter

load = lambda name: json.load(open(f"{SAVE_DIR}/{name}", encoding="utf-8"))
openie   = load(f"openie_results_ner_{LLM_NAME.replace('/', '_')}.json")
with_sch = load("initial_memory_with_schema.json")
filtered = load("memory_after_ontology_filtering.json")
final    = load("memory.json")
conflict = load("conflict_detection_results.json")
resolve  = load("conflict_resolution_results.json")
mgraph   = load("graph_from_memory/memory_graph.json")

def layer_sizes(m):
    return {k: len(m.get(k, [])) for k in ("schema_layer", "fact_layer", "passage_layer")}

stages = pd.DataFrame({
    "OpenIE + ontoloji": layer_sizes(with_sch),
    "Ontoloji filtresi sonrası": layer_sizes(filtered),
    "Çelişki çözümü sonrası (final)": layer_sizes(final),
}).rename(index={"schema_layer": "Şema", "fact_layer": "Olgu", "passage_layer": "Pasaj"})
display(stages)

print("Çelişki tespiti :", {k: conflict["summary"][k] for k in ("num_detection_tasks", "num_conflict_items", "total_tokens")})
print("Çelişki çözümü  :", resolve["summary"]["actions_applied"], "| olgu:", resolve["summary"]["facts_before"], "→", resolve["summary"]["facts_after"])
print("Bellek grafı    :", mgraph["stats"])

### 8.2 Katmanlar arası bağlantılara örnek: şema → olgular → pasaj

In [ ]:
schemas  = sorted(final["schema_layer"], key=lambda s: -s["frequency"])
facts    = {f["idx"]: f for f in final["fact_layer"]}
passages = {p["idx"]: p for p in final["passage_layer"]}

for schema in schemas[:5]:
    print(f"ŞEMA  {tuple(schema['content'])}   (frekans={schema['frequency']})")
    for fi in schema["fact_indices"][:3]:
        fact = facts[fi]
        print(f"   OLGU  {tuple(fact['content'])}")
        for pi in fact["passage_indices"][:1]:
            print(f"      PASAJ  {passages[pi]['content'][:110]}...")
    print()

### 8.3 Tespit edilen çelişkilere ve çözüm kararlarına örnek

In [ ]:
items = conflict["conflict_items"]
print(f"{len(items)} çelişki çifti bulundu.\n")
for item in items[:5]:
    print(f"[{item['conflict_type']}, güven={item['confidence']}]")
    print("  ", item["target_triple"], "\n  ", item["other_triple"])
    print("   Gerekçe:", str(item["reason"])[:200], "\n")
actions = resolve.get("applied_actions", [])
if actions:
    display(pd.DataFrame(actions)[["fact_id", "action", "resolved_triple", "reason"]].head(10))

### 8.4 Graf görselleştirmesi: bir destek pasajı ve çevresi (pasaj → varlıklar → ilişkili varlıklar → tipler)

In [ ]:
import re
import networkx as nx
import matplotlib.pyplot as plt

norm_text = lambda s: re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()
nodes_by_id = {n["id"]: n for n in mgraph["nodes"]}
adj = {}
for e in mgraph["edges"]:
    adj.setdefault(e["source"], []).append(e)

questions = json.load(open("data/hotpotqa_small/questions.json", encoding="utf-8"))
title = norm_text(questions[0]["supporting_facts"][0][0])
passages_with_title = [n["id"] for n in mgraph["nodes"] if n["layer"] == "passage" and title in norm_text(n["content"])]
passage_id = max(passages_with_title or [n["id"] for n in mgraph["nodes"] if n["layer"] == "passage"],
                 key=lambda pid: sum(e["type"] == "passage_entity" for e in adj.get(pid, [])))

G = nx.Graph()
G.add_node(passage_id, layer="passage", label="PASAJ\n" + nodes_by_id[passage_id]["content"].split(":", 1)[-1][:30] + "...")
entities = [e["target"] for e in adj.get(passage_id, []) if e["type"] == "passage_entity"][:12]
for ent in entities:
    G.add_node(ent, layer="entity", label=nodes_by_id[ent]["label"][:25])
    G.add_edge(passage_id, ent, type="passage_entity")
    related = [e for e in adj.get(ent, []) if e["type"] == "entity_relation" and e["target"] not in entities][:2]
    for e in related + [e for e in adj.get(ent, []) if e["type"] == "entity_to_type"]:
        layer = nodes_by_id[e["target"]]["layer"]
        G.add_node(e["target"], layer=layer, label=nodes_by_id[e["target"]]["label"][:25])
        G.add_edge(ent, e["target"], type=e["type"])

colors = {"passage": "#eb6834", "entity": "#2a78d6", "type": "#1baf7a"}
names = {"passage": "Pasaj", "entity": "Varlık", "type": "Tip (şema)"}
plt.figure(figsize=(13, 9))
pos = nx.spring_layout(G, seed=3, k=1.2)
for layer, color in colors.items():
    nodelist = [n for n in G if G.nodes[n]["layer"] == layer]
    nx.draw_networkx_nodes(G, pos, nodelist=nodelist, node_color=color, label=names[layer],
                           node_size=900 if layer == "passage" else 380, node_shape="s" if layer == "type" else "o")
nx.draw_networkx_edges(G, pos, alpha=0.35)
nx.draw_networkx_labels(G, pos, nx.get_node_attributes(G, "label"), font_size=8)
plt.title(f"Destek pasajı: '{questions[0]['supporting_facts'][0][0]}'  ({G.number_of_nodes()} düğüm)")
plt.legend(frameon=False, markerscale=0.6); plt.axis("off"); plt.show()

## 9. Erişim (Retrieval) ve Soru-Cevap

README'deki `code/retrieval_dataset_test.py` komutu çalıştırılıyor. README'deki ayarlar korunuyor (`--skip-fact-rerank true`, `--fact-similarity-threshold 0.4`). Klasik RAG ile adil karşılaştırma için LLM'e verilen pasaj sayısı her iki yöntemde de 5 (`--qa-top-k 5`).

In [ ]:
!python code/retrieval_dataset_test.py \
  --questions data/hotpotqa_small/questions.json \
  --save-dir {SAVE_DIR} \
  --output results/hotpotqa_small/memgraphrag_qa.json \
  --llm-name {LLM_NAME} \
  --llm-base-url {LLM_BASE_URL} \
  --embedding-model {EMBEDDING_MODEL} \
  --sample-num 0 --qa-top-k 5 --qa-max-workers 4 \
  --skip-fact-rerank true \
  --fact-similarity-threshold 0.4 \
  --use-raw-threshold-filter true 2>&1 | grep -vE "httpx|it/s\]|s/it\]|^\s*$"

### 9.1 Sonuçlar

Metrikler:
- **EM (Exact Match):** Normalize edilmiş tahmin, altın cevapla birebir aynı mı?
- **F1:** Tahmin ile altın cevap arasındaki kelime örtüşmesi (kısmi doğruluk).
- **Destek Recall@5:** Sorunun destek paragraf başlıklarından kaçı LLM'e verilen ilk 5 parçada geçiyor (erişim kalitesi için yaklaşık bir ölçü).

In [ ]:
import re, string
from collections import Counter

def normalize(s):
    s = str(s).lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def em(pred, gold):
    return float(normalize(pred) == normalize(gold))

def f1(pred, gold):
    p, g = normalize(pred).split(), normalize(gold).split()
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0:
        return 0.0
    prec, rec = common / len(p), common / len(g)
    return 2 * prec * rec / (prec + rec)

def support_recall(docs, supporting_facts, k=5):
    # Parçalar tokenizer'dan geçtiği için küçük harfli ve noktalama etrafı boşluklu; iki taraf da sadeleştirilir.
    simplify = lambda s: " " + re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip() + " "
    titles = {t for t, _ in supporting_facts}
    text = simplify(" ".join(docs[:k]))
    return sum(simplify(t) in text for t in titles) / len(titles)

mem = json.load(open("results/hotpotqa_small/memgraphrag_qa.json", encoding="utf-8"))
print("Özet:", json.dumps(mem["summary"], indent=1, ensure_ascii=False))

mem_df = pd.DataFrame([{
    "Soru": r["question"], "Altın cevap": r["answer"], "MemGraphRAG cevabı": r["predicted_answer"],
    "EM": em(r["predicted_answer"], r["answer"]), "F1": round(f1(r["predicted_answer"], r["answer"]), 2),
    "Recall@5": support_recall(r["retrieved_docs"], r["supporting_facts"]),
} for r in mem["results"]])
pd.set_option("display.max_colwidth", 80)
display(mem_df)
print(mem_df[["EM", "F1", "Recall@5"]].mean().round(3).to_dict())

### 9.2 Bir sorunun ayrıntılı incelemesi (getirilen pasajlar ve modelin akıl yürütmesi)

In [ ]:
r = mem["results"][0]
print("SORU      :", r["question"])
print("ALTIN     :", r["answer"])
print("DESTEK    :", [t for t, _ in r["supporting_facts"]])
print("\nGETİRİLEN İLK 5 PARÇA:")
for doc, score in zip(r["retrieved_docs"][:5], r["doc_scores"][:5]):
    print(f"  [{score:.3f}] {doc[:150]}...")
print("\nMODELİN HAM CEVABI:\n", r["raw_response"])

## 10. Klasik RAG ile Karşılaştırma

Aynı korpus, aynı embedding modeli (bge-large), aynı LLM (Qwen3.5-9B) ve aynı soru-cevap istemi (prompt) kullanılarak **klasik (yoğun vektör, dense) RAG** çalıştırıldı.
Tek fark erişim adımı: graf ve PageRank kullanılmadan, soru embedding'ine en benzer 5 parça getiriliyor (`MemGraphRAG.dense_passage_retrieval`, MemGraphRAG'ın da olgu bulamadığında kullandığı yedek yöntem).

In [ ]:
import sys, logging
sys.path.insert(0, "/content/MemGraphRAG/code")
logging.getLogger().setLevel(logging.WARNING)
from src.MemGraphRAG import MemGraphRAG
from src.utils.config_utils import BaseConfig
from src.utils.misc_utils import QuerySolution

config = BaseConfig(save_dir=SAVE_DIR, llm_name=LLM_NAME, llm_base_url=LLM_BASE_URL,
                    embedding_model_name=EMBEDDING_MODEL, openie_mode="online",
                    force_index_from_scratch=False, qa_top_k=5, qa_max_workers=4, max_new_tokens=1000)
rag = MemGraphRAG(global_config=config)
rag.prepare_retrieval_objects()

qs = [q["question"] for q in questions]
rag.get_query_embeddings(qs)
dense_solutions = []
for q in qs:
    ids, scores = rag.dense_passage_retrieval(q)
    docs = [rag.chunk_embedding_store.get_row(rag.passage_node_keys[i])["content"] for i in ids[:5]]
    dense_solutions.append(QuerySolution(question=q, docs=docs, doc_scores=scores[:5]))
dense_solutions, dense_raw, _ = rag.qa(dense_solutions)

dense_df = pd.DataFrame([{
    "Soru": q["question"], "Altın cevap": q["answer"], "Klasik RAG cevabı": s.answer,
    "EM": em(s.answer, q["answer"]), "F1": round(f1(s.answer, q["answer"]), 2),
    "Recall@5": support_recall(s.docs, q["supporting_facts"]),
} for q, s in zip(questions, dense_solutions)])
display(dense_df)

In [ ]:
summary = pd.DataFrame({
    "Klasik RAG (dense)": dense_df[["EM", "F1", "Recall@5"]].mean(),
    "MemGraphRAG": mem_df[["EM", "F1", "Recall@5"]].mean(),
}).T.round(3)
display(summary)

# Soru tipine göre kırılım (bridge = köprü/çok adımlı, comparison = karşılaştırma)
types = pd.Series([q["type"] for q in questions], name="Tip")
display(pd.concat({"Klasik RAG": dense_df[["EM", "F1", "Recall@5"]].groupby(types).mean(),
                   "MemGraphRAG": mem_df[["EM", "F1", "Recall@5"]].groupby(types).mean()}, axis=1).round(3))

import numpy as np
metrics = ["EM", "F1", "Recall@5"]
x = np.arange(len(metrics)); w = 0.36
fig, ax = plt.subplots(figsize=(7, 4))
for i, (name, color) in enumerate([("Klasik RAG (dense)", "#eb6834"), ("MemGraphRAG", "#2a78d6")]):
    bars = ax.bar(x + (i - 0.5) * w, summary.loc[name, metrics], w - 0.02, label=name, color=color)
    ax.bar_label(bars, fmt="%.2f", fontsize=9, color="#52514e", padding=2)
ax.set_xticks(x, metrics); ax.set_ylim(0, 1.1); ax.set_ylabel("Ortalama skor")
ax.set_title(f"HotpotQA alt kümesi ({len(questions)} soru) – LLM: {LLM_NAME}")
ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.25); ax.set_axisbelow(True)
ax.legend(frameon=False); plt.tight_layout(); plt.show()

### 10.1 İki yöntemin farklı cevap verdiği sorular

In [ ]:
cmp = pd.DataFrame({"Soru": mem_df["Soru"], "Altın": mem_df["Altın cevap"],
                    "Klasik RAG": dense_df["Klasik RAG cevabı"], "MemGraphRAG": mem_df["MemGraphRAG cevabı"],
                    "F1 farkı (Mem − Klasik)": (mem_df["F1"] - dense_df["F1"]).round(2)})
display(cmp[cmp["F1 farkı (Mem − Klasik)"] != 0])

## 11. Karşılaşılan Hatalar ve Çözümleri

| # | Hata / Sorun | Nedeni | Çözüm |
|---|---|---|---|
| 1 | `pip install -r requirements.txt` Colab'ın torch sürümünü 2.5.1'e düşürmeye çalışıyor, `vllm` derlemesi uzun sürüyor ve CUDA uyumsuzluğu uyarıları veriyor | Gereksinim dosyası yazarların kendi sunucusuna göre sabitlenmiş; vllm yalnızca offline OpenIE için gerekli | Colab'ın torch'u korunup yalnızca kodun import ettiği paketler kuruldu (Bölüm 3.3) |
| 2 | `ModuleNotFoundError: No module named 'gritlm'` | `embedding_model/__init__.py` GritLM sınıfını koşulsuz import ediyor (bge kullansak bile) | `gritlm` paketi kurulum listesine eklendi |
| 3 | README'deki `datasets/corpus/corpus.txt` ve `small-questions.json` bulunamadı | Repodaki klasör adı `dataset/`; README'deki örnek dosyalar repoya eklenmemiş | Repodaki `dataset/hotpotqa` verisinden korpus ve soru dosyası üretildi (Bölüm 6) |
| 4 | `code/run_index.sh` çalıştırılınca `PYTHON: unbound variable` hatası | Betik `set -u` ile tanımsız `$PYTHON` değişkenini kullanıyor; içinde yazarın yerel yolları (`/home/wcj/...`) var | Betik yerine `python code/index.py ...` komutu doğrudan, kendi yollarımızla çalıştırıldı |
| 5 | Colab'da Ollama kurulumu `zstd` bulunamadığı için yarıda kalıyor | Ollama'nın güncel Linux paketi `zstd` ile sıkıştırılmış; Colab imajında bu araç yok | Kurulumdan önce `apt-get install zstd` çalıştırıldı (Bölüm 4.1) |
| 6 | Uzun istemler hata vermeden kesiliyor, model eksik bağlamla cevap üretiyor | Ollama'nın varsayılan bağlam uzunluğu 4096 token; çelişki tespiti istemi varsayılan ayarlarla ~12.000 token'a çıkabiliyor | `OLLAMA_CONTEXT_LENGTH=8192` ayarlandı ve `--conflict-max-related 5` ile istem kısaltıldı (Bölüm 4.2, 7) |
| 7 | Tek bir geçici hata (sunucu meşgul, zaman aşımı) tüm indekslemeyi durdurabiliyor (`index_with_memory failed during schema extraction`) | `CacheOpenAI.infer` içinde tekrar deneme yok; varsayılan 20 paralel işçi, T4'ün kaldırabileceğinden fazla | `tenacity` ile üstel beklemeli tekrar deneme eklendi; paralel işçi sayısı Ollama ile eşit olacak şekilde 4'e indirildi (Bölüm 5, 7) |
| 8 | Tek bir bozuk JSON cevabı tüm bellek oluşturmayı iptal ediyor | `_memory_json` onaramadığı JSON'da istisna fırlatıyor | Onarılamayan cevap uyarı ile `{}` döndürülüp atlanıyor (Bölüm 5) |
| 9 | `RuntimeError: index_with_memory failed during OpenIE: division by zero` | LLM çağrıları başarısız olunca (Ollama kapalı, model yüklenmemiş) OpenIE hataları yalnızca uyarı olarak loglanıyor; hiç varlık çıkmayınca ortalama hesabı sıfıra bölünüyor ve asıl hata gizleniyor | Model, indekslemeden önce Bölüm 4.4'teki test hücresiyle doğrulandı. Bu hata görülürse loglardaki `WARNING ... openie` satırlarına ve `/content/ollama.log` dosyasına bakılmalı |
| 10 | Qwen3.5'in Turing mimarili GPU'larda (T4 dahil) çökebildiği raporlanmış | Ollama/llama.cpp tarafındaki bir hata ([ollama#14715](https://github.com/ollama/ollama/issues/14715)) | Bölüm 4.4'te GPU üzerinde test; hata durumunda yedek model Gemma 4 E4B |

<!-- Colab çalıştırması sırasında ek hata çıkarsa buraya satır olarak eklenecek -->

## 12. Teknik Değerlendirme: MemGraphRAG, Klasik RAG ve GraphRAG

### Klasik RAG
Belgeler parçalara bölünür, her parça embedding'e çevrilir; soruya en benzer *k* parça LLM'e verilir.
- **Güçlü yanı:** Basit, ucuz, indeksleme sırasında LLM çağrısı yok.
- **Zayıf yanı:** Her parçayı bağımsız değerlendirir. Çok adımlı sorularda ("X filminin yıldızı neyle tanınır?") ikinci adımdaki paragraf soruya yüzeysel olarak benzemediği için getirilmeyebilir. Parçalar arasındaki ilişki ve çelişki bilgisi yoktur.

### GraphRAG (Microsoft GraphRAG, HippoRAG vb.)
LLM ile metinden varlık–ilişki grafı çıkarılır; erişim graf üzerinden yapılır (topluluk özetleri ya da HippoRAG'daki gibi Kişiselleştirilmiş PageRank).
- **Güçlü yanı:** Varlıklar üzerinden parçalar arası bağlantı kurar, çok adımlı sorularda ilgili ama yüzeysel olarak benzemeyen pasajlara ulaşabilir.
- **Zayıf yanı:** Graf doğrudan ham OpenIE çıktısından kurulur. LLM'in çıkardığı gürültülü, tekrarlı veya **birbiriyle çelişen** üçlüler grafa olduğu gibi girer ve PPR'yi yanlış yöne çekebilir. Varlıkların tip bilgisi (kişi, film, şehir...) yoktur.

### MemGraphRAG'ı farklı kılan noktalar
1. **Üç katmanlı bellek:** Graf doğrudan değil, *şema–olgu–pasaj* belleğinden türetilir. Her olgunun kaynağı (pasajı) ve soyut tipi (şeması) izlenebilir.
2. **Ontoloji (şema) çıkarımı ve filtreleme:** Olgular tiplerle soyutlanır, nadir şemalar gürültü kabul edilip elenir. Grafa **tip düğümleri** eklenir; aynı tipteki varlıklar dolaylı olarak birbirine bağlanır.
3. **Çelişki farkındalığı:** Olgular arası katı çelişkiler pasaj kanıtıyla tespit edilir, bağlı çelişki grupları LLM ile çözülür (koru/düzelt/at). Klasik RAG ve standart GraphRAG'da bu adım yoktur.
4. **Maliyet:** Bu temizlik ücretsiz değil. Olgu başına ontoloji ve çelişki çağrıları yüzünden indeksleme maliyeti HippoRAG'dan da yüksektir. Bu deneyde ≈40 paragraf için yüzlerce LLM çağrısı yapıldı ve T4'te dakikalar sürdü (süre Bölüm 7'de); klasik RAG'da indeksleme sırasında hiç LLM çağrısı yok. Sorgulama aşamasında ise ek LLM maliyeti yok (`--skip-fact-rerank true`), yalnızca PPR hesaplanıyor.

### Bu deneyden gözlemler
_(Bu bölüm Bölüm 10'daki sonuçlara göre doldurulacak: EM/F1/Recall farkları, bridge ve comparison sorularındaki fark, tespit edilen çelişki sayısı, indeksleme süresi ve token maliyeti.)_

**Sınırlılıklar:** 10 soru istatistiksel olarak küçük bir örneklemdir; korpus küçük olduğu için klasik RAG'ın doğru parçayı bulma şansı da yüksektir. Farkların büyük ölçekli korpuslarda (makaledeki gibi binlerce paragraf ve çok sayıda çeldirici) daha belirgin olması beklenir.